# **Import Library**


In [ ]:
# Umum
import os
import keras
import numpy as np
from glob import glob 
from tqdm import tqdm 
import tensorflow as tf

# Data
import pandas as pd
from keras.preprocessing.image import ImageDataGenerator

# Visualisasi Data
import seaborn as sns
import matplotlib.pyplot as plt

# Model 
from keras import Sequential
from keras.layers import GlobalAvgPool2D, Dense, Dropout
from keras.models import load_model

# Model Pre-Trained
from tensorflow.keras.applications import ResNet50, ResNet50V2, InceptionV3, Xception

# Callbacks 
from keras.callbacks import EarlyStopping, ModelCheckpoint

# **Data**


In [ ]:
class_names = sorted(os.listdir('../input/coffee-bean-dataset-resized-224-x-224/train'))
n_classes = len(class_names)
class_names

Ada **4 Kelas** dalam **Dataset** ini


In [ ]:
class_dis = [len(glob("../input/coffee-bean-dataset-resized-224-x-224/train/" + name + "/*.png")) for name in class_names]
class_dis

In [ ]:
data = pd.read_csv('../input/coffee-bean-dataset-resized-224-x-224/Coffee Bean.csv')
data.head()

Muat Gambar

In [ ]:
# Inisialisasi Generator
train_gen = ImageDataGenerator(
    rescale=1./255, 
    horizontal_flip=True, 
    rotation_range=20, 
    validation_split=0.2)

test_gen = ImageDataGenerator(rescale=1./255)

# Muat Data
path = "../input/coffee-bean-dataset-resized-224-x-224/"
train_ds = train_gen.flow_from_directory(path + "train", target_size=(256,256), shuffle=True, batch_size=32, subset="training", class_mode='binary')
valid_ds = train_gen.flow_from_directory(path + "train", target_size=(256,256), shuffle=True, batch_size=32, subset="validation", class_mode='binary')
test_ds = train_gen.flow_from_directory(path + "test", target_size=(256,256), shuffle=True, batch_size=32, class_mode='binary')


# **Visualisasi Data**


In [ ]:
def show_image(img, title=None):
    plt.imshow(img)
    plt.title(title)
    plt.axis('off')

In [ ]:
plt.figure(figsize=(10,8))
i=1
for images, labels in iter(train_ds):
    
    id = np.random.randint(len(images))
    image = images[id]
    label = labels[id]

    plt.subplot(5,4,i)
    show_image(image, title=class_names[int(label)])

    i+=1
    if i>=21: break

plt.show()

# **Transfer Learning**

## **ResNet50V2**


In [ ]:
# Inisialisasi Model Dasar
name = "ResNet50V2"
base_model = ResNet50V2(include_top=False, weights='imagenet', input_shape=(256,256,3))
base_model.trainable = False

# Model
resnet_50V2 = Sequential([
    base_model, 
    GlobalAvgPool2D(),
    Dense(256, activation='relu'),
    Dropout(0.2),
    Dense(n_classes, activation='softmax')
])


# Compile
resnet_50V2.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Callbacks
cbs = [EarlyStopping(patience=3, restore_best_weights=True), ModelCheckpoint(name + ".h5", save_best_only=True)]

# Training
resnet_50V2.fit(train_ds, validation_data=valid_ds, epochs=50, callbacks=cbs)

In [ ]:
model_path = '../input/coffee-bean-classifier-resnet50v2/ResNet50V2-Coffee-Beans.h5'
model = load_model(model_path)

In [ ]:
model.summary()

In [ ]:
model.evaluate(test_ds)

# **Evaluasi**


In [ ]:
plt.figure(figsize=(15,20))
i=1
for images, labels in iter(test_ds):
    
    # Muat Gambar dan Label secara Acak
    id = np.random.randint(len(images))
    image = images[id]
    label = labels[id]

    # Buat Prediksi
    pred_label = class_names[int(np.argmax(model.predict(image[np.newaxis,...])))]

    # Plot Prediksi
    plt.subplot(5,4,i)
    show_image(image, title=f"Asli : {class_names[int(label)]} Prediksi : {pred_label}")

    # Increment dan Berhenti
    i+=1
    if i>=21: break

# Tampilkan
plt.show()
